# Chapter 5 — Functions

Source: *Effective Python*, 3rd Edition (Brett Slatkin), Chapter 5.

---


### Item 30 — Know That Function Arguments Can Be Mutated

Python passes arguments "by object reference" — a mutable argument (list, dict, custom object) passed into a function can be modified in place by that function, and the caller *will* see the change after the call returns, since both names point to the same underlying object. This is often exactly what you want (e.g., a function that appends to a list you handed it) but it's a footgun if unintended — the caller has no syntactic hint that their object might come back changed.


In [1]:
def add_default_tags(tags):
    tags.append('untagged')   # mutates the SAME list object the caller passed in
    return tags

my_tags = ['python', 'notebook']
result = add_default_tags(my_tags)

print(result)
print(my_tags)              # the caller's original list changed too -- same object!
print(result is my_tags)    # True -- not a copy

# If mutation isn't intended, defend by copying first
def add_default_tags_safe(tags):
    tags = list(tags)   # work on a copy -- caller's original list is untouched
    tags.append('untagged')
    return tags

original = ['python']
safe_result = add_default_tags_safe(original)
print(original, safe_result)


['python', 'notebook', 'untagged']
['python', 'notebook', 'untagged']
True
['python'] ['python', 'untagged']


### Item 31 — Return Dedicated Result Objects Instead of Requiring Function Callers to Unpack More Than Three Variables

A function returning a tuple of 2-3 values (`min_val, max_val = get_range(...)`) is fine, but beyond three, positional unpacking becomes error-prone — it's easy to misorder or miscount values on the caller's side, with no error until something downstream breaks. Once you're past three return values, return a `@dataclass` (or `NamedTuple`) instead: the caller accesses fields by name, which can't be silently misordered.


In [2]:
from dataclasses import dataclass

def compute_stats_tuple(numbers):
    return min(numbers), max(numbers), sum(numbers) / len(numbers), sum(numbers)   # 4 values -- risky

lo, hi, avg, total = compute_stats_tuple([4, 8, 15, 16, 23, 42])   # easy to misorder these 4 names
print(lo, hi, avg, total)

@dataclass
class Stats:
    minimum: float
    maximum: float
    average: float
    total: float

def compute_stats(numbers):
    return Stats(
        minimum=min(numbers),
        maximum=max(numbers),
        average=sum(numbers) / len(numbers),
        total=sum(numbers),
    )

stats = compute_stats([4, 8, 15, 16, 23, 42])
print(stats.minimum, stats.average)   # accessed by NAME -- can't be silently misordered


4 42 18.0 108
4 18.0


### Item 32 — Prefer Raising Exceptions to Returning `None`

A function that returns `None` to signal "failed" is ambiguous the moment `None` could also be a *valid* successful result (or if the caller simply forgets to check for it — `None` passes silently through most code until it causes a confusing error far from the real problem). Raising a specific exception instead forces the caller to either handle the failure explicitly or let it propagate loudly — it can never be silently ignored.


In [3]:
def safe_divide_none(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None   # ambiguous: is None "division failed" or "the answer happens to be None"?

result = safe_divide_none(10, 0)
if result is None:                 # caller MUST remember to check -- easy to forget
    print('Handled: division failed')

def safe_divide_raises(a, b):
    if b == 0:
        raise ValueError(f'Cannot divide {a} by zero')
    return a / b

try:
    safe_divide_raises(10, 0)
except ValueError as e:
    print('Forced to handle:', e)   # impossible to silently ignore an exception the same way


Handled: division failed
Forced to handle: Cannot divide 10 by zero


### Item 33 — Know How Closures Interact with Variable Scope and `nonlocal`

A nested function can *read* variables from its enclosing function's scope automatically (a closure) — but assigning to that variable inside the nested function creates a brand-new **local** variable instead of modifying the outer one, unless you explicitly declare `nonlocal variable_name` first. This is one of Python's most common closure surprises.


In [4]:
def make_counter_broken():
    count = 0
    def increment():
        count += 1   # this is actually `count = count + 1` -- creates a NEW local `count`!
        return count
    return increment

counter = make_counter_broken()
try:
    counter()
except UnboundLocalError as e:
    print('UnboundLocalError:', e)

def make_counter_fixed():
    count = 0
    def increment():
        nonlocal count   # explicitly says: modify the ENCLOSING scope's count, not a new local one
        count += 1
        return count
    return increment

counter2 = make_counter_fixed()
print(counter2(), counter2(), counter2())


UnboundLocalError: local variable 'count' referenced before assignment
1 2 3


### Item 34 — Reduce Visual Noise with Variable Positional Arguments

`def f(*args)` lets a function accept any number of positional arguments, collected into a tuple — useful when the caller has a variable-length list of values to pass and you don't want to force them to build a list/tuple explicitly just to hand it over. Pair it with `*` at the call site (`f(*my_list)`) to unpack an existing sequence into separate positional arguments.


In [5]:
def log(message, *values):     # *values collects any extra positional args into a tuple
    if not values:
        print(message)
    else:
        formatted = ', '.join(str(v) for v in values)
        print(f'{message}: {formatted}')

log('My numbers are', 1, 2, 3)   # caller doesn't need to build a list manually
log('Hi there')                   # zero extra args is fine too -- values is just an empty tuple

my_values = [7, 8, 9]
log('Unpacked from a list', *my_values)   # * at the CALL SITE unpacks a sequence into positional args


My numbers are: 1, 2, 3
Hi there
Unpacked from a list: 7, 8, 9


### Item 35 — Provide Optional Behavior with Keyword Arguments

Keyword arguments (`def f(a, b=10)`) let callers skip optional parameters entirely, and — used at the call site (`f(a=1, b=2)`) — make each argument's meaning explicit regardless of order, which is far more readable than a long list of positional values where you have to count positions to know what each one means.


In [6]:
def create_user(name, age=18, country='unknown', active=True):
    return {'name': name, 'age': age, 'country': country, 'active': active}

# Positional: works, but you must remember the exact order of every field
print(create_user('Ada', 30, 'UK', True))

# Keyword: explicit, order-independent, and callers can skip what they don't need to change
print(create_user('Alan', country='UK'))
print(create_user(name='Grace', active=False, age=25))   # any order, all self-documenting


{'name': 'Ada', 'age': 30, 'country': 'UK', 'active': True}
{'name': 'Alan', 'age': 18, 'country': 'UK', 'active': True}
{'name': 'Grace', 'age': 25, 'country': 'unknown', 'active': False}


### Item 36 — Use `None` and Docstrings to Specify Dynamic Default Arguments

Default argument values are evaluated **once**, when the function is defined — not each time it's called. This is a serious trap for mutable defaults (`def f(items=[])`) since every call without an explicit argument shares the *same* list object, accumulating state across unrelated calls. The fix: default to `None`, then create the actual mutable value fresh inside the function body, documenting the real default behavior in the docstring.


In [7]:
def append_to_broken(item, target=[]):   # DANGER: [] is created ONCE, at definition time
    target.append(item)
    return target

print(append_to_broken('a'))   # ['a']
print(append_to_broken('b'))   # ['a', 'b'] -- the SAME list leaked across calls!

def append_to_safe(item, target=None):
    """Append item to target, or to a fresh list if target is None (default: new list each call)."""
    if target is None:
        target = []   # a genuinely NEW list every time this branch runs
    target.append(item)
    return target

print(append_to_safe('a'))   # ['a']
print(append_to_safe('b'))   # ['b'] -- independent, as expected


['a']
['a', 'b']
['a']
['b']


### Item 37 — Enforce Clarity with Keyword-Only and Positional-Only Arguments

A bare `*` in a function signature marks everything after it as **keyword-only** (must be passed by name — prevents callers from accidentally relying on positional order for less-obvious parameters). A bare `/` marks everything before it as **positional-only** (can't be passed by name — useful when the parameter name is an implementation detail that might change later).


In [8]:
def resize(image, /, width, *, height, keep_aspect=True):
    #        ^-- positional-only        ^-- keyword-only, must be named
    return f'resizing to {width}x{height}, keep_aspect={keep_aspect}'

print(resize('photo.png', 800, height=600))            # image is positional, height is keyword
print(resize('photo.png', width=800, height=600))       # width can ALSO be passed positionally or by name

try:
    resize(image='photo.png', width=800, height=600)   # image is positional-ONLY -- this fails
except TypeError as e:
    print('TypeError:', e)

try:
    resize('photo.png', 800, 600)   # height is keyword-ONLY -- can't be passed positionally
except TypeError as e:
    print('TypeError:', e)


resizing to 800x600, keep_aspect=True
resizing to 800x600, keep_aspect=True
TypeError: resize() got some positional-only arguments passed as keyword arguments: 'image'
TypeError: resize() takes 2 positional arguments but 3 were given


### Item 38 — Define Function Decorators with `functools.wraps`

A decorator normally wraps a function inside another function, which means the wrapped result loses the original function's `__name__`, `__doc__`, and other metadata (tools like debuggers and help() then show the wrapper's identity, not the real function's). `functools.wraps(original_func)`, applied to the inner wrapper, copies that metadata across automatically — always use it when writing a decorator.


In [9]:
import functools

def trace_broken(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f'{func.__name__}(...) -> {result!r}')
        return result
    return wrapper   # no functools.wraps -- metadata is lost

@trace_broken
def add(a, b):
    """Add two numbers."""
    return a + b

print(add(2, 3))
print('name became:', add.__name__)   # 'wrapper', not 'add' -- metadata lost!

def trace_fixed(func):
    @functools.wraps(func)   # preserves func's __name__, __doc__, etc. on the wrapper
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f'{func.__name__}(...) -> {result!r}')
        return result
    return wrapper

@trace_fixed
def multiply(a, b):
    """Multiply two numbers."""
    return a * b

print(multiply(2, 3))
print('name preserved:', multiply.__name__, '-', multiply.__doc__)


add(...) -> 5
5
name became: wrapper
multiply(...) -> 6
6
name preserved: multiply - Multiply two numbers.


### Item 39 — Prefer `functools.partial` over `lambda` Expressions for Glue Functions

When you need to "pre-fill" some of a function's arguments to match an interface another piece of code expects (a callback, a `sorted(key=...)`, etc.), a `lambda` works, but `functools.partial(func, *fixed_args, **fixed_kwargs)` states the intent more directly — "this is `func` with some arguments already bound" — and it preserves useful introspection (like being able to inspect the original function) that a lambda wrapper obscures.


In [10]:
from functools import partial

def power(base, exponent):
    return base ** exponent

# lambda glue: works, but is an anonymous wrapper with no real identity of its own
square_lambda = lambda x: power(x, 2)
print(square_lambda(5))

# functools.partial: states directly "power, with exponent already fixed to 2"
square_partial = partial(power, exponent=2)
print(square_partial(5))
print(square_partial.func, square_partial.keywords)   # introspectable: which func, which args are fixed

cube = partial(power, exponent=3)
print(cube(4))
print(list(map(square_partial, [1, 2, 3, 4])))   # reads clearly in a map() call too


25
25
<function power at 0xffa6016a8670> {'exponent': 2}
64
[1, 4, 9, 16]
